In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_milo
# python -m ipykernel install --user --name scrna_cartography_milo --display-name "milo"

In [ ]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# milo
import pertpy as pt
milo = pt.tl.Milo()
import mudata as md
from mudata import MuData

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict
# plotting
import matplotlib.pyplot as plt 

import random
from sklearn.metrics.pairwise import euclidean_distances
from sklearn_ann.kneighbors.annoy import AnnoyTransformer 

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import diff_genes as dg
import misc as mi
import dg_milo as dm

In [ ]:
# Create directories
mi.create_directories(dir_path = str(here('data/differential_abundance/files')))
mi.create_directories(dir_path = str(here('data/differential_abundance/de_analysis_groups')))
mi.create_directories(dir_path = str(here('data/differential_abundance/de_analysis_overall')))
mi.create_directories(dir_path = str(here('data/differential_abundance/plots')))
mi.create_directories(dir_path = str(here('data/differential_abundance/objects')))

In [ ]:
# Paths
base_dir = str(here('data/differential_abundance/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
de_dir = os.path.join(base_dir, 'de_analysis_groups') 
de_overall_dir = os.path.join(base_dir, 'de_analysis_overall') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

#### Load data

In [ ]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

In [ ]:
mdata = md.read_h5mu(os.path.join(objects_dir,"mdata_666_annotation.h5mu"))

#### MILO analysis

In [ ]:
mdata = milo.load(adata)
sc.pp.neighbors(mdata["rna"], use_rep="X_latent_1", transformer=AnnoyTransformer(666))
milo.make_nhoods(mdata, prop=0.1)
mdata = milo.count_nhoods(mdata, sample_col="ic_id_platform_adjusted_sample")
mdata["rna"].obs["sample_cell_count"] = (
    mdata["rna"].obs.groupby("ic_id_platform_adjusted_sample")["ic_id_platform_adjusted_sample"]
    .transform("count")
)
milo.annotate_nhoods(mdata, anno_col="manual_annotation")
mdata["milo"].var["nhood_annotation_origianl"] = mdata["milo"].var["nhood_annotation"]
mdata["milo"].var["nhood_annotation"] = mdata["milo"].var["nhood_annotation"].cat.add_categories("mixed")
mdata["milo"].var.loc[mdata["milo"].var["nhood_annotation_frac"] < 0.5, "nhood_annotation"] = "mixed"
mdata.write(os.path.join(objects_dir,"mdata_666_annotation.h5mu"))

In [ ]:
ref_cells = mdata["rna"].obs_names[mdata["rna"].obs["nhood_ixs_refined"] == 1] # Get reference cells, in neighbourhood order
nhood_to_ref = dict(enumerate(ref_cells)) # dictionary with nhood number 
nhoods = mdata["rna"].obsm["nhoods"] # Get the neighborhoods
cell_names = mdata["rna"].obs_names # Get the cell names
rows, cols = nhoods.nonzero() # numeric neighbourhood id (column index)
df = pd.DataFrame({
    "cell": cell_names[rows],
    "nhood_id": cols,                    # numeric
    "neighborhood": ref_cells[cols],     # barcode
})
df = df.merge(adata.obs[['disease_harmonized', 'ic_id_platform_adjusted_sample', 'manual_annotation']].reset_index().rename(columns={"index": "cell"}), how = "left", on = "cell")
df = df.merge(mdata['milo'].var.rename(columns={"index_cell": "neighborhood"}), how = "left", on = "neighborhood")

#### Check that more than 1 sample is in a neighborhood

In [ ]:
# 1) samples per neighborhood
samples_per_nhood = df.groupby("neighborhood")["ic_id_platform_adjusted_sample"].nunique()
assert samples_per_nhood.median() > 5, "Too few samples per neighborhood"

# 2) condition mixing
conds_per_nhood = df.groupby("neighborhood")["disease_harmonized"].nunique()
assert (conds_per_nhood >= 2).mean() > 0.9, "Too many neighborhoods lack condition mixing"

# 3) neighborhood size
nhood_sizes = df.groupby("neighborhood").size()
assert nhood_sizes.min() > 50, "Neighborhoods too small"
assert nhood_sizes.max() < 5000, "Neighborhoods too large"

# 4) sample dominance
df_counts = (
    df.groupby(["neighborhood", "ic_id_platform_adjusted_sample"])
    .size()
    .reset_index(name="n")
)

df_counts["frac"] = df_counts.groupby("neighborhood")["n"].transform(lambda x: x / x.sum())
max_frac = df_counts.groupby("neighborhood")["frac"].max()

assert max_frac.median() < 0.8, "Neighborhoods dominated by single sample"

conds_per_nhood = df.groupby("neighborhood")["disease_harmonized"].nunique()
assert (conds_per_nhood >= 2).mean() > 0.9, "Too many neighborhoods lack condition mixing"

# Annotation specificity 
assert (mdata["milo"].var["nhood_annotation_frac"] > 0.8).mean() > 0.9, "Too many neighborhoods are specific to one cell type"

print("All sanity checks passed.")

In [ ]:
df.to_csv(os.path.join(files_dir, "cells_in_neighborhood.csv"))

#### Differential abundance

In [ ]:
mdata["rna"].obs["log_sample_cell_count"] = np.log1p(
    mdata["rna"].obs["sample_cell_count"]
)

In [ ]:
# pre vs nd without covariate
dm.milo_da_nhoods(mdata, design="~disease_harmonized",
    model_contrasts="disease_harmonizedpre-disease_harmonizednd")
pre_nocov = mdata["milo"].var.copy()

# t2d vs nd without covariate
dm.milo_da_nhoods(mdata, design="~disease_harmonized",
    model_contrasts="disease_harmonizedt2d-disease_harmonizednd")
t2d_nocov = mdata["milo"].var.copy()

#### Save

In [ ]:
t2d_nocov.to_csv(os.path.join(files_dir, "t2d_vs_nd.csv"))
pre_nocov.to_csv(os.path.join(files_dir, "pre_vs_nd.csv"))